# From-scratch VLM: log-sum-exp pooled layers as image patches + question + answer text

A small vision-language model with every weight randomly initialized. The 29 saved Qwen2.5-7B answer-token states are log-sum-exp pooled into 15 layer windows (same pooling as the 4 × 128 MLP run) and treated as 15 image patches: a vision encoder (patch embedding + layer positions + 2 transformer layers) encodes them, an MLP projector maps them into the multimodal space, and a 4-layer single-stream fusion transformer attends over [CLS] + layer tokens + question + [SEP] + generated answer. A linear head on [CLS] predicts BLEURT-correct vs hallucinated. Text uses the Qwen tokenizer's vocabulary remapped to source-train tokens; embeddings are learned from scratch.

Same saved six-source split, BCE loss, 40 epochs, batch 128, AdamW, seed 42, best-source-validation-AUROC checkpoint selection, threshold calibration and held-out TriviaQA evaluation as the MLP runs (learning rate 3e-4 with warmup + cosine for the transformer).

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 2ad8a2e
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_scratch_vlm_logsumexp')
command = [sys.executable, '/kaggle/working/SALA/sala_vlm_scratch.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--batch-size', '128']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Parameters:', summary['parameters'], 'text vocab:', summary['text_vocab_size'], 'best epoch:', summary['best_epoch'])
print('Source val AUROC:', summary['best_source_validation_auroc_during_training'])
print('Source test:', json.dumps(summary['source_test_metrics'], indent=2))
print('Target (TriviaQA):', json.dumps(summary['target_metrics'], indent=2))
display(results.head())